In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from transformers import AutoTokenizer
import pandas as pd
from typing import Tuple, List, Dict, Optional

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import re
import ast
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import time


from pathlib import Path




# Cargar dataset con noticias

In [29]:
df_news = pd.read_csv('subds_con_tickers_para_fase2_v2.csv')

In [30]:
df_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42267 entries, 0 to 42266
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Date                 42267 non-null  object
 1   Article_title        42267 non-null  object
 2   Stock_symbol         42267 non-null  object
 3   Url                  42267 non-null  object
 4   Article              42267 non-null  object
 5   texto_full           42267 non-null  object
 6   tickers_encontrados  42267 non-null  object
dtypes: object(7)
memory usage: 2.3+ MB


In [31]:
df_news.head(2)

,Date,Article_title,Stock_symbol,Url,Article,texto_full,tickers_encontrados
0,2023-11-30 00:00:00+00:00,UK antitrust regulator wins appeal over Apple ...,AAPL,https://www.nasdaq.com/articles/uk-antitrust-r...,Adds details from ruling and CMA comment in pa...,UK antitrust regulator wins appeal over Apple ...,['AAPL']
1,2023-11-30 00:00:00+00:00,Japan aircon king Daikin looks to custom chips...,AAPL,https://www.nasdaq.com/articles/japan-aircon-k...,"By Sam Nussey and Miho Uranaka\nTOKYO, Dec 1 (...",Japan aircon king Daikin looks to custom chips...,"['AAPL', 'AMZN']"


# Detección del sentimiento financiero con Finbert

A partir de la lectura del paper __FinBERT: Financial Sentiment Analysis with Pre-trained Language Models__ decidimos seleccionar el modelo de [ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert) disponible en Hugging Face.

Este modelo está pre-entrenado para analizar el sentimiento financiero de una noticia. Para hacerlo, se debe contar con un Token de Hugging Face. 

De acuerdo a lo leído en la documentación que comparten, sabemos que el modelo recibe una noticia y devuelve softmax outputs para tres labels:
- Sentimiento positivo
- Sentimiento neutro
- Sentimiento negativo

[readme del modelo pre-entrenado para detección de sentimientos financieros](https://huggingface.co/ProsusAI/finbert/blob/main/README.md)


Adicional a la información específica del modelo, la página de Hugging Face provee snippets de código que nos sirvieron como base para el armado de los casos de análisis individuales y del pipeline "automatizado" que se muestra más adelante en el notebook.

**Ej de un snippet de código:**

```
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

result = client.text_classification(
    "I like you. I love you",
    model="ProsusAI/finbert",
)
```



## La limitación de los tokens

Lamentablemente, al comenzar a trabajar, nos encontramos con que el modelo de Finbert solo es capas de analizar textos de 512 tokens, lo cual fue una limitante importante porque la gran mayoría de nuestras noticias poseían una cantidad de tokens mayor.

Para evaluar posibles vías de acción decidimos entender cuantos chunks de 510 tokens tenían nuestras noticias (el proceso genera tokens adicionales y 510 fue el número más alto que logramos que analice por request). Para esto, las partimos en dichos chunks y elaboramos histogramas para cada acción que nos permitan ver esta distribución.

Encaramos varias pruebas, sin embargo (y muy lamentablemente) consumimos los tokens gratuitos disponibles en Hugging Face antes de poder finalizarlas.

A continuación comentamos brevemente las pruebas, pero decidimos incluir en el código únicamente aquella que pudimos completar y con la que decidimos avanzar en el proyecto, a fin de que este quede más limpio.

### Histogramas del número de chunks en los que se requiere particionar las noticias para cada stock symbol 

In [33]:
# Contamos los chuncks por noticia
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

def contar_chunks(texto, max_length = 510, stride = 50):
    if not isinstance(texto, str) or not texto.strip():
        return 0
    enc = tokenizer(
        texto,
        return_overflowing_tokens=True, #si el texto supera max_length, genere varios trozos (chunks) consecutivos con solapamiento
        truncation=True,
        max_length=max_length,
        stride=stride,
        add_special_tokens=True,
    )
    return len(enc["input_ids"])

# Agregar columna n_chunks
df_news["n_chunks"] = df_news["Article"].apply(contar_chunks)


In [41]:
#Expandir el dataframe: 1 fila por ticker

def parse_tickers(value):
    return ast.literal_eval(value)


tmp = df_news[["tickers_encontrados", "n_chunks"]].copy()
tmp["ticker"] = tmp["tickers_encontrados"].apply(parse_tickers)
tmp = tmp.explode("ticker")

#Orden de aparición por frecuencia
ticker_counts = tmp["ticker"].value_counts()
tickers = ticker_counts.index.tolist()


In [47]:
# Grafico
# Parámetros
COLS = 3
TITLE = "Distribución de n_chunks por ticker"

# Crear subplots
rows = math.ceil(len(tickers) / COLS)
fig = make_subplots(
    rows=rows,
    cols=COLS,
    subplot_titles=tickers,
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

# Histograma por ticker
for i, tk in enumerate(tickers):
    r = i // COLS + 1
    c = i % COLS + 1

    data_tk = tmp.loc[tmp["ticker"] == tk, "n_chunks"]
    x_max = int(data_tk.max())

    fig.add_trace(
        go.Histogram(
            x=data_tk,
            xbins=dict(start=0.5, end=x_max + 0.5, size=1),
            name=tk,
            marker_color="#eb990c",
            showlegend=False,
        ),
        row=r,
        col=c,
    )

    fig.update_xaxes(title_text="n_chunks", dtick=3, row=r, col=c, tickangle = 0)


# 6) Layout final
fig.update_layout(
    title=TITLE,
    bargap=0.05,
    height=max(350, 300 * rows),
    width=max(600, 320 * min(COLS, len(tickers))),
)

fig.show()

Como se puede ver, la mayoría de las noticias tienen 3 chuncks.

Nuestra intención inicial fue procesar todas las noticias de 5 chuncks o menos (luego disminuimos a 3 chuncks o menos, debido al tiempo de procesamiento), enviando los chunks al modelo, sin embargo, nos encontramos con el siguiente problema:
- Para una misma noticia, los puntajes de los diferentes chunks eran muy diversos. Era frecuente, por ejemplo, que en el primer chunk en puntaje postivo fuese cercano a 1 y el negativo a 0 y en el siguiente esto se invirtiera. Viendo esto, no consideramos que avanzar de este modo y luego tomar, por ejemplo, promedios tuviera sentido, ya que considerábamos que modelo perdía la capacidad de evaluar el sentimiento general del texto al recibirlo en porciones.

Evaluamos la posibilidad de usar un modelo local con conocimiento financiero (por ejemplo: martain7r/finance-llama-8b:fp16) para resumir el texto a menos de 500 tokns y luego obtener el sentimiento financiero del resumen, sin embargo, desistimos de esta opción por los tiempos de procesamiento que nos estaba llevando.

A raíz de estas complicaciones, decidimos avanzar sólo con las noticias de 512 tokens o menos (con caracteres especiales incluidos). Somos concientes de que esta opción no es la más representativa, pero creemos que a fin del análisis comparativo que buscamos hacer, puede proveernos insights valiosos, al menos en esta primera versión del proyecto.

## Obtencion del sentimiento financiero con Finbert para las noticias de 512 tokens o menos

In [ ]:
# Load environment variables from a .env file in this folder (or parents)
load_dotenv()

# Read token from environment
token = os.getenv("TOKEN_HF")

client = InferenceClient(token=token)


In [49]:
df_news_1_chunk = df_news.loc[df_news["n_chunks"] == 1].copy()
print(f"Filtradas {len(df_news_1_chunk)} noticias (1 chunk) de un total de {len(df_news)}.")

Filtradas 5918 noticias (1 chunk) de un total de 42267.


Importante --> los siguientes bloques de código los armamos para poder evaluar las noticias con hasta 5 chuncks, por eso pueden parecer más complejo de los necesario.

In [ ]:
MAX_LEN = 510
STRIDE = 50
LABELS = ("positive", "neutral", "negative")


def finbert_scores_per_chunk(
    text: str,
    client,
    tokenizer,
    max_length: int = MAX_LEN,
    stride: int = STRIDE,
    cap_chunks: Optional[int] = 1,   # limitado a 1 chunk
):
    """
    Devuelve una lista de dicts por chunk con keys: positive, neutral, negative.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    enc = tokenizer(
        text,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
    )
    input_ids_batches = enc["input_ids"]
    chunks = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids_batches]
    if cap_chunks is not None:
        chunks = chunks[:cap_chunks]

    out: List[Dict[str, float]] = []
    for ch in chunks:
        outputs = client.text_classification(ch, model="ProsusAI/finbert")
        scores = {"positive": 0.0, "neutral": 0.0, "negative": 0.0}
        for o in outputs:
            label = (getattr(o, "label", None) or o.get("label", "")).lower()
            score = float(getattr(o, "score", None) or o.get("score", 0.0))
            if label in scores:
                scores[label] = score
        out.append(scores)
    return out

def agg_scores_mean(per_chunk: List[Dict[str, float]]) -> Dict[str, float]:
    """
    Promedio simple de los scores por chunk (si no hay chunks, NaN).

    Imp --> esta era sólo una alternativa que estábamos evaluando para el trabajo con las noticias partidas en chunks.
            Dado que no avanzamos con esta vía, no exploramos más opciones, pero no estamos convencidas (al menos por los 
            datos que llegamos a evaluar, que el promedio hubiera sido la mejor alternativa).
    """
    if not per_chunk:
        return {"positive": np.nan, "neutral": np.nan, "negative": np.nan}
    n = len(per_chunk)
    pos = sum(d["positive"] for d in per_chunk) / n
    neu = sum(d["neutral"]  for d in per_chunk) / n
    neg = sum(d["negative"] for d in per_chunk) / n
    return {"positive": pos, "neutral": neu, "negative": neg}

def get_chunk_val(chunks: List[Dict[str, float]], idx: int, key: str):
    try:
        return chunks[idx].get(key, np.nan)
    except Exception:
        return np.nan



In [ ]:
df_news_1_chunk["Date"] = pd.to_datetime(df_news_1_chunk["Date"])

df_news_1_chunk.sort_values(by = "Date", ascending = True, inplace = True)

In [ ]:
OUTPUT_OK = Path('../outputs/df_news_1_chunk_con_sentimiento_v1_incremental.csv')
OUTPUT_ERR = Path('../outputs/df_news_1_chunk_errores_v1.csv')

# Columnas nuevas que van a ser agregadas al CSV de resultados
chunk_cols = []
for i in range(1, 4):
    chunk_cols += [f'pos_chunk_{i}', f'neu_chunk_{i}', f'neg_chunk_{i}']

new_result_cols = (
    chunk_cols
    + ['pos_total', 'neu_total', 'neg_total', 'sentiment_label', 'n_actual_chunks', 'row_idx']
)

# Orden de columnas: todas las originales del df + las nuevas
orig_cols = list(df_news_1_chunk.columns)
header_order = orig_cols + [c for c in new_result_cols if c not in orig_cols]

def append_dict_as_row_csv(row_dict: dict, csv_path: Path, header_order: list | None = None):
    """
    Agrega un dict como fila a un CSV en modo append.
    Escribe header si el archivo no existe o está vacío.
    Si se provee header_order, reordena/agrega columnas faltantes antes de escribir.
    Importante --> Dado que esto tardo varios días en correr, trabajar con esta escritura incremental
    nos dió respaldo frente a posibles errores o interrupciones en el proceso
    """
    df = pd.DataFrame([row_dict])
    if header_order is not None:
        # Asegura que existan todas las columnas del header y las ordena
        for k in header_order:
            if k not in df.columns:
                df[k] = pd.NA
        df = df[header_order]
    write_header = (not csv_path.exists()) or (csv_path.stat().st_size == 0)
    df.to_csv(csv_path, mode='a', index=False, header=write_header, encoding='utf-8')

def load_already_processed_indices(csv_path: Path) -> set[int]:
    """
    Lee row_idx del CSV de resultados (si existe) para poder reanudar
    sin reprocesar filas ya guardadas.
    """
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return set()
    try:
        # Solo leemos la columna identificadora mínima para evitar cargar todo el archivo
        s = pd.read_csv(csv_path, usecols=['row_idx'])
        return set(s['row_idx'].tolist())
    except Exception:
        # Si no existe la columna (primera ejecución, etc.)
        return set()

In [ ]:
# Verificaciones previas mínimas
#(De nuevo, este código puede parecer un poco redundante, pero el objetivo era que nos ayudara en casos de fallas)
assert 'df_news_1_chunk' in globals(), "Falta df_news_1_chunk"
assert 'client' in globals(), "Falta client (InferenceClient)"
assert 'tokenizer' in globals(), "Falta tokenizer (AutoTokenizer)"
assert 'finbert_scores_per_chunk' in globals(), "Falta finbert_scores_per_chunk"
assert 'agg_scores_mean' in globals(), "Falta agg_scores_mean"

# Cargar índices ya procesados para reanudar
already_done = load_already_processed_indices(OUTPUT_OK)
print(f"Filas ya procesadas (según CSV): {len(already_done)}")

# Función para armar el dict de salida de una fila
def build_output_row(row_idx, row, per_chunk: list[dict], totals: dict):
    out = row.to_dict()  # todas las columnas originales
    # Cargar scores por chunk (hasta 5) con NaN si faltan
    def safe_chunk(i, key):
        try:
            return per_chunk[i].get(key, np.nan)
        except Exception:
            return np.nan

    for i in range(1):
        out[f'pos_chunk_{i+1}'] = safe_chunk(i, 'positive')
        out[f'neu_chunk_{i+1}'] = safe_chunk(i, 'neutral')
        out[f'neg_chunk_{i+1}'] = safe_chunk(i, 'negative')

    # Totales
    out['pos_total'] = totals.get('positive', np.nan)
    out['neu_total'] = totals.get('neutral', np.nan)
    out['neg_total'] = totals.get('negative', np.nan)

    # Etiqueta final por argmax si hay valores
    # Esta era una pureba que hicimos para seleccionar una label de sentimiento única para
    # la noticia, pero finalmente no la utilizamos
    if not all(pd.isna(v) for v in [out['pos_total'], out['neu_total'], out['neg_total']]):
        out['sentiment_label'] = max(
            {'positive': out['pos_total'], 'neutral': out['neu_total'], 'negative': out['neg_total']},
            key=lambda k: {'positive': out['pos_total'], 'neutral': out['neu_total'], 'negative': out['neg_total']}[k]
        )
    else:
        out['sentiment_label'] = None

    # Número de chunks efectivamente usados
    n_eff = len(per_chunk) if isinstance(per_chunk, list) else 0
    out['n_actual_chunks'] = max(1, n_eff)

    # Identificador de reanudación
    out['row_idx'] = int(row_idx)
    return out

# Bucle principal
total = len(df_news_1_chunk)
start_time = time.time()
processed = 0
errors = 0

for row_idx, row in df_news_1_chunk.iterrows():
    # Reanudar: saltear los ya presentes en el CSV
    if int(row_idx) in already_done:
        continue

    try:
        # 1) Scores por chunk
        per_chunk_scores = finbert_scores_per_chunk(
            row['texto_full'],
            client=client,
            tokenizer=tokenizer,
            cap_chunks=1
        )

        # 2) Totales
        totals = agg_scores_mean(per_chunk_scores)

        # 3) Armar fila de salida (original + nuevas columnas)
        out_row = build_output_row(row_idx, row, per_chunk_scores, totals)

        # 4) Guardar esa fila en el CSV final (append)
        append_dict_as_row_csv(out_row, OUTPUT_OK, header_order=header_order)

        processed += 1

        # Visualización del progreso del proceso
        if processed % 20 == 0:
            elapsed = time.time() - start_time
            print(f"[{processed} nuevas filas guardadas | {errors} errores] - {elapsed/60:.1f} min transcurridos")

    except Exception as e:
        # Registrar el error y seguir
        err_row = {
            'row_idx': int(row_idx),
            'error_type': type(e).__name__,
            'error_msg': str(e)[:1000],  # truncado
        }
        # Data para guardar en los registros que retornen errores
        for col in ['id', 'url', 'titulo', 'fecha', 'tickers_encontrados', 'n_chunks', 'n_tokens_con_especiales']:
            if col in df_news_1_chunk.columns:
                err_row[col] = row.get(col, None)

        append_dict_as_row_csv(err_row, OUTPUT_ERR)
        errors += 1
        # Seguir con la siguiente fila (no interrumpe el proceso!!)
        continue

elapsed = time.time() - start_time
print(f"Listo. Nuevas filas guardadas: {processed} | Errores: {errors} | Tiempo: {elapsed/60:.1f} min")
print(f"CSV resultados: {OUTPUT_OK}")
print(f"CSV errores:    {OUTPUT_ERR}")

Filas ya procesadas (según CSV): 0
[20 nuevas filas guardadas | 0 errores] - 0.2 min transcurridos
[40 nuevas filas guardadas | 0 errores] - 0.4 min transcurridos
[60 nuevas filas guardadas | 0 errores] - 0.6 min transcurridos
[80 nuevas filas guardadas | 0 errores] - 0.9 min transcurridos
[100 nuevas filas guardadas | 0 errores] - 1.1 min transcurridos
[120 nuevas filas guardadas | 0 errores] - 1.2 min transcurridos
[140 nuevas filas guardadas | 0 errores] - 1.4 min transcurridos
[160 nuevas filas guardadas | 0 errores] - 1.5 min transcurridos
[180 nuevas filas guardadas | 0 errores] - 1.7 min transcurridos
[200 nuevas filas guardadas | 0 errores] - 1.9 min transcurridos
[220 nuevas filas guardadas | 0 errores] - 2.1 min transcurridos
[240 nuevas filas guardadas | 0 errores] - 2.3 min transcurridos
[260 nuevas filas guardadas | 0 errores] - 2.5 min transcurridos
[280 nuevas filas guardadas | 0 errores] - 2.6 min transcurridos
[300 nuevas filas guardadas | 0 errores] - 2.8 min transcur

In [ ]:
done = load_already_processed_indices(OUTPUT_OK)
pendientes = [i for i in df_news_1_chunk.index if int(i) not in done]

print(f"Total df_news_1_chunk: {len(df_news_1_chunk)}")
print(f"Ya procesadas (en CSV): {len(done)}")
print(f"Pendientes: {len(pendientes)}")

if OUTPUT_ERR.exists() and OUTPUT_ERR.stat().st_size > 0:
    df_err = pd.read_csv(OUTPUT_ERR)
    print(f"Errores registrados: {len(df_err)}")
    display(df_err.head(5))

Total df_news_1_chunk: 5918
Ya procesadas (en CSV): 5918
Pendientes: 0
